In [1]:
from transformers import AutoModelForMaskedLM

In [2]:
model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [3]:
parameters = model.num_parameters() / 1_000_000
print(f"The model has {parameters:.2f} million parameters")

The model has 66.99 million parameters


In [78]:
text = "This is a great [MASK]."

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


In [79]:
inputs = tokenizer(text, return_tensors="pt")
token_ids = tokens["input_ids"][0]
print(token_ids)
token_strings = tokenizer.convert_ids_to_tokens(token_ids)
print(token_strings)    
decoded_text = tokenizer.decode(tokens["input_ids"][0])
print(decoded_text)


tensor([ 101, 2023, 2003, 1037, 2307,  103, 1012,  102])
['[CLS]', 'this', 'is', 'a', 'great', '[MASK]', '.', '[SEP]']
[CLS] this is a great [MASK]. [SEP]


In [81]:
import torch
inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits

mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
print(mask_token_logits.shape)

top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    word = tokenizer.decode([token])
    new_sentence = text.replace(tokenizer.mask_token, word)
    print(new_sentence)

torch.Size([1, 30522])
This is a great deal.
This is a great success.
This is a great adventure.
This is a great idea.
This is a great feat.


In [83]:
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [92]:
size_in_mb = dataset["train"].dataset_size / (1024 * 1024)
print(size_in_mb)
print(dataset["train"].shape)

127.03209114074707
(25000, 2)
